In [2]:
import os
import sys

# Change to project root directory
os.chdir('..')  # Move from notebooks/ to qanda-v2/
sys.path.append('.')  # Add current directory to Python path
print("Working directory:", os.getcwd())

# Now imports should work
from qanda_module.config import setup_system
from qanda_module.data_processing import get_semantic_topic_matches, build_panelist_text_profile
from sentence_transformers import SentenceTransformer

# Now the config paths should work correctly
helpers, qa_chain, config = setup_system("config.yaml")
embedder = SentenceTransformer(config.embedding_model_name)
print("✅ Setup complete!")

Working directory: /Users/swang/workspace/repos/qanda_clean1/qanda-v2
Loaded 475 panelist profile URLs
Total panellist responses: 17815
Panellist responses with missing speaker_id: 0
✅ Setup complete!


In [3]:
# If person exists but URL is None/NaN
# Check your dim_panellist table:
import pandas as pd
paterson_data = helpers.df_guests[helpers.df_guests['name'].str.contains('PATERSON', na=False)]
print(paterson_data[['name', 'profession', 'link']])

Empty DataFrame
Columns: [name, profession, link]
Index: []


In [5]:
import pandas as pd
paterson_data = helpers.df_guests[helpers.df_guests['name'].str.contains('Paterson', na=False)]
print(paterson_data[['name', 'profession', 'link']])

               name                    profession  \
42   James Paterson                                 
61   James Paterson                                 
146  James Paterson                                 
405  James Paterson                                 
550  James Paterson  Liberal Senator for Victoria   
553  James Paterson  Liberal Senator for Victoria   
610  James Paterson  Liberal Senator for Victoria   

                                                  link  
42   https://www.abc.net.au/qanda/james-paterson/10...  
61   https://www.abc.net.au/qanda/james-paterson/10...  
146  https://www.abc.net.au/qanda/james-paterson/10...  
405  https://www.abc.net.au/qanda/james-paterson/10...  
550  https://www.abc.net.au/qanda/james-paterson/11...  
553  https://www.abc.net.au/qanda/james-paterson/11...  
610  https://www.abc.net.au/qanda/james-paterson/11...  


In [6]:
# Check what case your panelist_lookup uses
sample_names = list(helpers.panelist_lookup.keys())[:5]
print("Lookup keys:", sample_names)

# Check what case appears in a sample AI response
# (Run a quick query and see how names appear)

AttributeError: 'ImprovedQAHelpers' object has no attribute 'panelist_lookup'

In [7]:
test_panelists = ["TANYA PLIBERSEK", "MALCOLM TURNBULL", "PENNY WONG"]

debug_results = []
for panelist in test_panelists:
    try:
        profile = build_panelist_text_profile(helpers, panelist)
        profile_embedding = embedder.encode([profile])
        
        debug_results.append({
            'panelist': panelist,
            'profile_length': len(profile),
            'profile_preview': profile[:100],
            'embedding_shape': profile_embedding.shape,
            'has_nan': np.isnan(profile_embedding).any(),
            'has_inf': np.isinf(profile_embedding).any(),
            'all_zeros': np.allclose(profile_embedding, 0),
            'embedding_mean': np.mean(profile_embedding),
            'embedding_std': np.std(profile_embedding)
        })
    except Exception as e:
        debug_results.append({
            'panelist': panelist,
            'error': str(e)
        })

debug_df = pd.DataFrame(debug_results)

In [ ]:
debug_df

In [ ]:
# Cell 4: Investigate Topic Embeddings
from qanda_module.constants import ALL_SUBSTANTIVE_TOPICS

print(f"Number of topics: {len(ALL_SUBSTANTIVE_TOPICS)}")
print(f"Sample topics: {ALL_SUBSTANTIVE_TOPICS[:5]}")

# Generate topic embeddings
topic_embeddings = embedder.encode(ALL_SUBSTANTIVE_TOPICS)
print(f"Topic embeddings shape: {topic_embeddings.shape}")

# Check for problematic topic embeddings
topic_debug = pd.DataFrame({
    'topic': ALL_SUBSTANTIVE_TOPICS,
    'has_nan': [np.isnan(emb).any() for emb in topic_embeddings],
    'has_inf': [np.isinf(emb).any() for emb in topic_embeddings],
    'all_zeros': [np.allclose(emb, 0) for emb in topic_embeddings],
    'embedding_mean': [np.mean(emb) for emb in topic_embeddings],
    'embedding_std': [np.std(emb) for emb in topic_embeddings]
})

# Show any problematic topics
problem_topics = topic_debug[
    topic_debug['has_nan'] | topic_debug['has_inf'] | topic_debug['all_zeros']
]
print(f"Problematic topics: {len(problem_topics)}")
topic_debug.head(10)

In [ ]:
# Cell 5: Test Cosine Similarity Calculation
from sklearn.metrics.pairwise import cosine_similarity
import warnings

# Test with Tanya's profile
test_panelist = "TANYA PLIBERSEK"
profile = build_panelist_text_profile(helpers, test_panelist)
profile_embedding = embedder.encode([profile])
topic_embeddings = embedder.encode(ALL_SUBSTANTIVE_TOPICS)

print(f"Profile embedding shape: {profile_embedding.shape}")
print(f"Topic embeddings shape: {topic_embeddings.shape}")

# Check vector norms (cosine similarity divides by these)
profile_norm = np.linalg.norm(profile_embedding)
topic_norms = [np.linalg.norm(emb) for emb in topic_embeddings]

print(f"Profile norm: {profile_norm}")
print(f"Topic norms range: {min(topic_norms):.6f} to {max(topic_norms):.6f}")
print(f"Any zero norms: {any(norm == 0 for norm in topic_norms)}")

# Now test the actual cosine similarity with warnings captured
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    similarities = cosine_similarity(profile_embedding, topic_embeddings)[0]
    
    print(f"\nWarnings caught: {len(w)}")
    for warning in w:
        print(f"Warning: {warning.message}")

print(f"Similarities shape: {similarities.shape}")
print(f"Similarities range: {np.min(similarities):.6f} to {np.max(similarities):.6f}")
print(f"Any NaN in similarities: {np.isnan(similarities).any()}")

In [ ]:
# Cell 6: Test Episode-Related Functions
# Test the functions that might use 'date' column

# Test episode lookup
test_episodes = helpers.df_ep.head(3)
print("Sample episodes:")
print(test_episodes[['date', 'title']])

# Test episode-related helper functions
test_panelist = "TANYA PLIBERSEK"
try:
    episodes_text = helpers.get_panelist_episodes(helpers, test_panelist)
    print(f"✅ get_panelist_episodes worked")
except Exception as e:
    print(f"❌ get_panelist_episodes failed: {e}")

# Check if any other functions use 'date'
print("\nDataFrame columns:")
print("Episode columns:", helpers.df_ep.columns.tolist())

In [ ]:
# Cell 7: Investigate helpers object
print("Available methods on helpers:")
methods = [method for method in dir(helpers) if not method.startswith('_')]
for method in sorted(methods):
    print(f"  - {method}")

print(f"\nhelpers type: {type(helpers)}")

# Check if the method exists elsewhere
print(f"\nLooking for get_panelist_episodes...")
if hasattr(helpers, 'get_panelist_episodes'):
    print("✅ Method exists")
else:
    print("❌ Method missing!")

In [ ]:
# Cell 9: Test the fixed function
from qanda_module.ui_gradio import get_panelist_episodes

try:
    result = get_panelist_episodes(helpers, "TANYA PLIBERSEK")
    print("✅ Function works!")
    print(result)
except Exception as e:
    print(f"❌ Still broken: {e}")